# Lab 4.4 &mdash; MCP From the Wire Up

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Frame a JSON-RPC message the way MCP does, and find out why framing exists at all
- Write the server: initialize, tools/list, tools/call &mdash; the whole protocol surface you need
- Write the client, and watch discovery happen at run time rather than at build time
- Read an <code>mcpServers</code> config as what it is: a list of access grants

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **No SDK.** You implement the protocol, because a protocol you have implemented once
> is one you can reason about when it misbehaves. The last cell runs your server as a
> real subprocess over real pipes &mdash; and needs no model and no network.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the agent chooses, and then through tools it did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# The tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

MCP is JSON-RPC 2.0 in both directions over a transport. Over stdio there is no HTTP to tell the
reader where one message ends, so each is **framed** with a `Content-Length` header &mdash; the same
trick the Language Server Protocol uses, for the same reason.

Three methods carry almost everything:

| method | what it does |
|---|---|
| `initialize` | agree a protocol version and exchange capabilities |
| `tools/list` | **discovery** &mdash; the client learns the tools at run time |
| `tools/call` | invoke one by name with arguments |

Discovery is the part with consequences. The agent does not know what it can do until it asks,
which is what lets a server gain a tool without your redeploying &mdash; and what makes a server you
did not review a problem you did not review.

## Section 1 &mdash; Framing

One header, and the reason it must count **bytes** rather than characters.

In [ ]:
import re

def encode(message: dict) -> bytes:
    """Frame one JSON-RPC message for the stdio transport."""
    body = json.dumps(message).encode("utf-8")
    # TODO: the header that tells the reader exactly where this body ends.
    # Mind the units, and mind the blank line that ends the header block.
    header = BLANK
    return header + body


def decode_all(blob: bytes) -> list:
    """Every complete message in a byte stream -- which is what framing makes possible."""
    out, i = [], 0
    while True:
        j = blob.find(b"\r\n\r\n", i)
        if j < 0:
            return out
        n = int(re.search(r"Content-Length:\s*(\d+)", blob[i:j].decode("ascii")).group(1))
        start = j + 4
        out.append(json.loads(blob[start:start + n]))
        i = start + n

In [ ]:
# --- Self-check: Section 1
_m = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
_uni = {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
        "params": {"arguments": {"counterparty": "CAF\u00c9-EU"}}}

check("a message survives a round trip", lambda: decode_all(encode(_m)) == [_m])
check("the header names Content-Length",
      lambda: encode(_m).split(b"\r\n")[0].startswith(b"Content-Length:"))
check("the header block ends with a blank line",
      lambda: b"\r\n\r\n" in encode(_m))
check("two messages in one stream decode as two",
      lambda: decode_all(encode(_m) + encode(_m)) == [_m, _m])
check("the length counts BYTES, not characters",
      lambda: int(re.search(rb"Content-Length: (\d+)", encode(_uni)).group(1))
              == len(json.dumps(_uni).encode("utf-8")),
      "one non-ASCII character and a character count misreads every message after it")
check("and a non-ASCII payload still round-trips inside a stream",
      lambda: decode_all(encode(_uni) + encode(_m)) == [_uni, _m])

## Section 2 &mdash; The server

Note where tool failures go. A tool that could not do its job is a **successful** JSON-RPC
response carrying `isError: true` &mdash; because the protocol worked perfectly. A JSON-RPC `error`
means the *protocol* failed: unknown method, malformed request. Collapsing the two is the most
common MCP implementation bug, and it makes tool failures invisible to the model.

In [ ]:
PROTOCOL_VERSION = "2025-06-18"

TOOL_SPECS = [
    {"name": "lookup_payment",
     "description": lookup_payment.__doc__,
     "inputSchema": {"type": "object", "properties": {"ref": {"type": "string"}},
                     "required": ["ref"]}},
    {"name": "policy_for",
     "description": policy_for.__doc__,
     "inputSchema": {"type": "object", "properties": {"reason_code": {"type": "string"}},
                     "required": ["reason_code"]}},
]
SERVER_TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}


def _result(rid, payload):  return {"jsonrpc": "2.0", "id": rid, "result": payload}
def _rpc_error(rid, code, message):
    return {"jsonrpc": "2.0", "id": rid, "error": {"code": code, "message": message}}
def _tool_text(text, is_error=False):
    return {"content": [{"type": "text", "text": text}], "isError": is_error}


def handle(request: dict) -> dict:
    """One JSON-RPC request in, one response out. This is the entire server."""
    rid, method = request.get("id"), request.get("method")
    params = request.get("params") or {}

    if method == "initialize":
        return _result(rid, {"protocolVersion": PROTOCOL_VERSION,
                             "capabilities": {"tools": {}},
                             "serverInfo": {"name": "ledger", "version": "1.0.0"}})
    if method == "tools/list":
        return _result(rid, {"tools": TOOL_SPECS})
    if method == "tools/call":
        name = params.get("name")
        args = params.get("arguments") or {}
        fn = SERVER_TOOLS.get(name)
        if fn is None:
            return _result(rid, _tool_text(f"no such tool: {name!r}", is_error=True))
        # TODO: run it. A tool that fails is still a SUCCESSFUL response -- with isError set.
        # Use _tool_text(...) for both outcomes, and let nothing escape as an exception.
        return BLANK
    return _rpc_error(rid, -32601, f"method not found: {method}")

In [ ]:
# --- Self-check: Section 2
def _req(method, **params):
    return {"jsonrpc": "2.0", "id": 7, "method": method, "params": params}

check("initialize agrees a protocol version",
      lambda: handle(_req("initialize"))["result"]["protocolVersion"] == PROTOCOL_VERSION)
check("and names the server",
      lambda: handle(_req("initialize"))["result"]["serverInfo"]["name"] == "ledger")
check("tools/list exposes name, description and inputSchema for every tool",
      lambda: all({"name", "description", "inputSchema"} <= set(t)
                  for t in handle(_req("tools/list"))["result"]["tools"]))
check("the descriptions carried across are the real ones",
      lambda: "Not for searching" in handle(_req("tools/list"))["result"]["tools"][0]["description"])
check("a good call returns text content",
      lambda: "INSUFFICIENT_FUNDS" in handle(
          _req("tools/call", name="lookup_payment", arguments={"ref": "PMT-1002"})
      )["result"]["content"][0]["text"])
check("a good call is not flagged as an error",
      lambda: handle(_req("tools/call", name="lookup_payment",
                          arguments={"ref": "PMT-1002"}))["result"]["isError"] is False)
check("an unknown tool is a RESULT with isError, not a JSON-RPC error",
      lambda: handle(_req("tools/call", name="nope", arguments={}))["result"]["isError"] is True,
      "the protocol worked -- only the tool did not; collapsing these hides tool failures from the model")
check("a tool that raises is caught and reported as isError",
      lambda: handle(_req("tools/call", name="lookup_payment",
                          arguments={"wrong_arg": 1}))["result"]["isError"] is True)
check("nothing escapes the server as an exception",
      lambda: isinstance(handle(_req("tools/call", name="lookup_payment", arguments={})), dict))
check("an unknown METHOD is a real JSON-RPC error",
      lambda: handle(_req("tools/nonesuch"))["error"]["code"] == -32601,
      "this one really is a protocol failure, so it belongs in the error channel")

## Section 3 &mdash; The client, and discovery

The client below sends every message through `encode`/`decode_all`, so it is talking over the real
wire format even while the server is in the same process. Swapping in a pipe changes nothing above
the transport &mdash; which you prove in the last cell.

In [ ]:
class Session:
    """An MCP client session against one server."""

    def __init__(self, handler):
        self._handler, self._id, self.tools = handler, 0, {}

    def request(self, method: str, params: dict = None) -> dict:
        self._id += 1
        message = {"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}}
        [on_the_wire] = decode_all(encode(message))     # framed and parsed, as over a pipe
        return self._handler(on_the_wire)

    def initialize(self) -> dict:
        return self.request("initialize")["result"]

    def list_tools(self) -> dict:
        """Discovery. The client did not know these names until this call returned."""
        self.tools = {t["name"]: t for t in self.request("tools/list")["result"]["tools"]}
        return self.tools

    def call_tool(self, name: str, **arguments) -> dict:
        result = self.request("tools/call", {"name": name, "arguments": arguments})["result"]
        # TODO: MCP returns a LIST of content blocks. Take the text of the first one.
        text = BLANK
        return {"text": text, "is_error": bool(result.get("isError"))}

In [ ]:
# --- Self-check: Section 3
def _session():
    s = Session(handle)
    s.initialize()
    s.list_tools()
    return s

check("the session knows nothing about the tools before it asks",
      lambda: Session(handle).tools == {},
      "discovery at run time is what lets a server change without your redeploying")
check("and knows both of them afterwards",
      lambda: set(_session().tools) == {"lookup_payment", "policy_for"})
check("each request carries a fresh id",
      lambda: _session()._id == 2)
check("a tool call returns the text",
      lambda: "ZENITH" in _session().call_tool("lookup_payment", ref="PMT-1003")["text"])
check("and is not flagged as an error",
      lambda: _session().call_tool("lookup_payment", ref="PMT-1003")["is_error"] is False)
check("a failed call surfaces as is_error rather than an exception",
      lambda: _session().call_tool("nope")["is_error"] is True)
check("the second tool works through the same session",
      lambda: "Treasury approval" in
              _session().call_tool("policy_for", reason_code="LIMIT_BREACH")["text"])

def _show_discovery():
    s = _session()
    for name, spec in s.tools.items():
        print(f"  {name:16} {spec['description'].splitlines()[0][:64]}")
guard(_show_discovery)

## Section 4 &mdash; The config is the grant

Four lines of JSON give an agent a capability. Nothing in the agent's code changes, nothing is
compiled, and by default nothing reviews it. So read the file the way you would read an IAM policy:
**which of these entries lets the agent change something?**

In [ ]:
CONFIG = {
    "mcpServers": {
        "ledger":  {"command": "python", "args": ["-m", "ledger_mcp"],
                    "env": {"LEDGER_SCOPE": "read-only"}},
        "policy":  {"command": "python", "args": ["-m", "policy_mcp"],
                    "env": {"POLICY_SCOPE": "read-only"}},
        "release": {"command": "python", "args": ["-m", "release_mcp"],
                    "env": {"RELEASE_SCOPE": "write"}},
        "notes":   {"command": "python", "args": ["-m", "notes_mcp"]},
    }
}

WRITE_SCOPES = {"write", "read-write", "admin"}

def servers_that_can_write(config: dict) -> list:
    """The configured servers that grant the agent the power to change something."""
    out = []
    for name, entry in config["mcpServers"].items():
        scopes = {str(v).lower() for v in (entry.get("env") or {}).values()}
        if BLANK:                   # TODO: does this entry grant a write scope?
            out.append(name)
    return sorted(out)

In [ ]:
# --- Self-check: Section 4
_with_admin = {"mcpServers": {**CONFIG["mcpServers"],
                              "ops": {"command": "python", "args": ["-m", "ops_mcp"],
                                      "env": {"OPS_SCOPE": "admin"}}}}

check("exactly one configured server can write today",
      lambda: servers_that_can_write(CONFIG) == ["release"])
check("an admin scope is a write grant too",
      lambda: servers_that_can_write(_with_admin) == ["ops", "release"])
check("a server with no env declared is not treated as a write grant",
      lambda: "notes" not in servers_that_can_write(CONFIG))
check("the scope lives in the config, not in the agent's own code",
      lambda: all("SCOPE" in k
                  for e in CONFIG["mcpServers"].values() for k in (e.get("env") or {})),
      "which is what makes it reviewable and revocable without touching the agent")

def _grants():
    for name, entry in CONFIG["mcpServers"].items():
        env = entry.get("env") or {}
        print(f"  {name:9} {' '.join([entry['command']] + entry['args']):24} "
              f"{'WRITE' if name in servers_that_can_write(CONFIG) else 'read':>6}  {env}")
guard(_grants)

### The server you are about to launch

Small enough to read in a minute, which is the point. It is the same three methods, the same
framing, and its own private copy of a ledger &mdash; it shares nothing with this notebook.

In [ ]:
MCP_SERVER_SOURCE = r"""
import sys, json, re

LEDGER = {
    "PMT-1002": {"amount": 48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held", "reason_code": "LIMIT_BREACH"},
}
SPECS = [{"name": "lookup_payment",
          "description": "Return the ledger record for one payment reference such as PMT-1002.",
          "inputSchema": {"type": "object", "properties": {"ref": {"type": "string"}},
                          "required": ["ref"]}}]

def framed(msg):
    body = json.dumps(msg).encode("utf-8")
    return b"Content-Length: " + str(len(body)).encode() + b"\r\n\r\n" + body

def handle(req):
    rid, method = req.get("id"), req.get("method")
    params = req.get("params") or {}
    if method == "initialize":
        return {"jsonrpc": "2.0", "id": rid,
                "result": {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}},
                           "serverInfo": {"name": "ledger", "version": "1.0.0"}}}
    if method == "tools/list":
        return {"jsonrpc": "2.0", "id": rid, "result": {"tools": SPECS}}
    if method == "tools/call":
        ref = (params.get("arguments") or {}).get("ref")
        rec = LEDGER.get(ref)
        text = json.dumps({"ref": ref, **rec}) if rec else "no payment found with reference %r" % ref
        return {"jsonrpc": "2.0", "id": rid,
                "result": {"content": [{"type": "text", "text": text}], "isError": rec is None}}
    return {"jsonrpc": "2.0", "id": rid,
            "code": -32601, "error": {"code": -32601, "message": "method not found"}}

data, out, i = sys.stdin.buffer.read(), b"", 0
while True:
    j = data.find(b"\r\n\r\n", i)
    if j < 0:
        break
    n = int(re.search(rb"Content-Length:\s*(\d+)", data[i:j]).group(1))
    start = j + 4
    out += framed(handle(json.loads(data[start:start + n])))
    i = start + n
sys.stdout.buffer.write(out)
"""
print(f"{len(MCP_SERVER_SOURCE.splitlines())} lines of server")

## Run it for real &mdash; over a real pipe

No model and no network needed for this one. The cell writes a small MCP server to your work
directory, launches it as a **separate process**, and talks to it over stdin and stdout with the
framing you wrote in Section 1.

Everything above the transport is the same code. That is the claim the protocol makes, and this
is it being true.

In [ ]:
def talk_to_a_real_server():
    import subprocess, sys as _sys
    path = os.path.join(WORK, "ledger_mcp_server.py")
    with open(path, "w") as fh:
        fh.write(MCP_SERVER_SOURCE)

    payload = b"".join(encode(m) for m in [
        {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}},
        {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}},
        {"jsonrpc": "2.0", "id": 3, "method": "tools/call",
         "params": {"name": "lookup_payment", "arguments": {"ref": "PMT-1003"}}},
    ])
    proc = subprocess.run([_sys.executable, path], input=payload,
                          capture_output=True, timeout=60)
    if proc.returncode != 0:
        print("server exited", proc.returncode, proc.stderr.decode()[:300])
        return
    for msg in decode_all(proc.stdout):
        result = msg.get("result", {})
        if "serverInfo" in result:
            print(f"  initialize -> {result['serverInfo']} protocol {result['protocolVersion']}")
        elif "tools" in result:
            print(f"  tools/list -> discovered {[t['name'] for t in result['tools']]}")
        elif "content" in result:
            print(f"  tools/call -> {result['content'][0]['text'][:88]}")

guard(talk_to_a_real_server)

### Read it

That was a real process boundary: a separate interpreter, its own memory, its own environment, and
nothing shared with this notebook but two pipes. Give it different credentials and you have the
governance story from the deck &mdash; a tool you can grant, revoke and audit on its own.

You read that server before you ran it. Ask yourself what you actually know about a server you
install from a registry with one line of JSON &mdash; and carry the question into Module 8.

In [ ]:
score()

## Your turn

1. Add `resources/list` and `resources/read` to the server, and move `policy_for` behind a resource
   instead of a tool. Which agent behaviours become impossible &mdash; and is that a loss or the point?
2. The server answers requests in order and never initiates. Add a `notifications/tools/list_changed`
   message and decide what a client should do with it mid-run.
3. `talk_to_a_real_server` trusts the subprocess to be well behaved. Make the server emit a
   `Content-Length` that is 10 bytes too long, and watch `decode_all` wait forever for bytes that
   are not coming. Where does the timeout belong?